# ASL v1 — BIGGER pretrain (1500-class, ROI) + ROI fine-tune (Colab T4)

Re-trains the encoder on a **1500-gloss** ROI slice of ASL Citizen (vs the prior
500), then fine-tunes the 75-class head. Held-out val/test signers are excluded
from the pretrain set. Checkpoint/resume is enabled so a T4 disconnect doesn't
lose the run.

### One-time setup
1. `Runtime -> Change runtime type -> T4 GPU`.
2. Upload **three files** to `MyDrive/asl-model/`:
   - `code_bundle.zip`            (asl package + configs + manifests/norms)
   - `pretrain_1500_jpeg.npz`     (~3 GB - 1500-class ROI frames, JPEG-packed)
   - `clips_roi.npz`              (~1 GB - 75-class ROI fine-tune frames)

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

In [ ]:
import os, zipfile, shutil
DRIVE = '/content/drive/MyDrive/asl-model'
os.makedirs('/content/work/artifacts/cache', exist_ok=True)
with zipfile.ZipFile(f'{DRIVE}/code_bundle.zip') as z:
    z.extractall('/content/work')
for f in ['pretrain_1500_jpeg.npz', 'clips_roi.npz']:
    shutil.copy(f'{DRIVE}/{f}', f'/content/work/artifacts/cache/{f}')
%cd /content/work
!pip -q install pyyaml onnx onnxruntime
import torch; print('cuda available:', torch.cuda.is_available())

In [ ]:
# Decode the 1500-class ROI JPEG cache back to frames.dat
!PYTHONPATH=src python -m asl.pack_pretrain_jpeg --unpack \
    --in artifacts/cache/pretrain_1500_jpeg.npz --cache artifacts/cache/pretrain_1500

In [ ]:
# Pretrain encoder from scratch on the 1500-class ROI set. --resume continues
# from artifacts/checkpoints/pretrain/resume.pt if a prior session was cut off
# (harmless on a fresh run - the file won't exist yet).
!PYTHONPATH=src python -u -m asl.pretrain --cache artifacts/cache/pretrain_1500 \
    --norm artifacts/manifest/norm_roi.json \
    --epochs 45 --warmup 4 --batch-size 64 --lr 0.004 \
    --out artifacts/checkpoints/pretrain \
    --resume artifacts/checkpoints/pretrain/resume.pt
import shutil, os
os.makedirs('/content/drive/MyDrive/asl-model/out_1500', exist_ok=True)
shutil.copy('artifacts/checkpoints/pretrain/encoder.pt',
            '/content/drive/MyDrive/asl-model/out_1500/encoder.pt')
print('saved 1500-pretrain encoder.pt to Drive/out_1500')

In [ ]:
# Fine-tune the 75-class head on ROI clips + the 1500-pretrained encoder.
!PYTHONPATH=src python -u -m asl.train --config configs/finetune_roi.yaml
import shutil, os
OUT = '/content/drive/MyDrive/asl-model/out_1500'
for f in ['artifacts/checkpoints/finetune_roi/best.pt',
          'artifacts/checkpoints/finetune_roi/history.json']:
    shutil.copy(f, f'{OUT}/{os.path.basename(f)}')
print('1500 fine-tune done; best.pt + history.json copied to Drive/out_1500')